# 04c Taiwan Income Features

從台灣政府資料來源蒐集收入類特徵，對應 DeepSolar 資料集中的社經欄位，
以進行跨國遷移推論（Stage 2 模型的台灣推論）。

**地理單位**：鄉鎮市區（TOWNCODE）— 368 個，對應美國人口普查區  
**基準年份**：民國 109 年（2020），與人口資料對齊  
**輸出**：`data/taiwan/taiwan_income_features.csv`

## DeepSolar 欄位對應

| DeepSolar 欄位 | 台灣資料來源 | 說明 |
|---|---|---|
| `median_household_income` | 109年度綜稅綜合所得總額（村里） | 以納稅戶數加權彙整至鄉鎮 |
| `gini_index` | 同上（變異係數） | 標準差/平均數，代理所得不平等 |
| `poverty_family_below_poverty_level_rate` | 低收入戶戶數（2020 Q4）/ 總戶數 | 衛福部統計，低收入戶比例 |
| `poverty_family_below_poverty_level` | 低收入戶戶數（2020 Q4） | 衛福部統計，低收入戶絕對數量 |

In [ ]:
import sys
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
from scipy.special import erf

# ── font for Chinese labels ──────────────────────────────────────────────────
matplotlib.rcParams['font.family'] = ['Microsoft JhengHei', 'sans-serif']
matplotlib.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120

# ── paths ────────────────────────────────────────────────────────────────────
ROOT          = Path('../')
INCOME_CSV    = ROOT / 'data/taiwan/income/109年度綜稅綜合所得總額全國各縣市鄉鎮村里統計分析表.csv'
POVERTY_XLS   = ROOT / 'data/taiwan/income/1.1.2低收入戶戶數及人數按鄉鎮市區別分(2015~)1150310.xlsx'
POP_CSV       = ROOT / 'data/taiwan/population/taiwan_population_features.csv'
OUT_PATH      = ROOT / 'data/taiwan/income/taiwan_income_features.csv'
FIG_DIR       = ROOT / 'outputs/figures/transfer'
FIG_DIR.mkdir(parents=True, exist_ok=True)

print('Paths ready.')
print('Income CSV:', INCOME_CSV.resolve())
print('Poverty XLS:', POVERTY_XLS.resolve())

## Step 1 — 綜稅資料：村里 → 鄉鎮 彙整

**來源**：財政部 109 年度綜稅綜合所得總額全國各縣市鄉鎮村里統計分析表  
**用途**：`median_household_income`、`gini_index`  

> 注意：資料為**稅前綜合所得**，與 DeepSolar 的「可支配所得」定義差異約 10–15%，  
> 但相對排序（鄉鎮間高低）仍可用於模型推論。

**加權方式**：以各村里「納稅單位(戶)」作為權重，計算鄉鎮加權平均值。

**Gini 計算**：採用計量經濟學標準的**對數常態近似公式**，從變異係數（CV）換算為標準 Gini（0–1）：

$$\text{Gini} = \text{erf}\!\left(\frac{\sigma_{\log}}{2}\right), \quad \sigma_{\log} = \sqrt{\ln\!\left(1 + \left(\frac{CV}{100}\right)^2\right)}$$

此公式在所得服從對數常態分布的假設下為精確解，與 DeepSolar 的 Gini 係數定義一致。  
計算流程：村里層級 CV → 村里 Gini → 以戶數加權彙整至鄉鎮。

In [ ]:
income_raw = pd.read_csv(INCOME_CSV, encoding='utf-8-sig')
print('Raw shape:', income_raw.shape)
print('Columns:', income_raw.columns.tolist())
income_raw.head(3)

In [ ]:
def split_township(combined: str):
    """Split '臺北市松山區' → ('臺北市', '松山區')."""
    m = re.match(r'^(.*?[市縣])(.*)', str(combined))
    return (m.group(1), m.group(2)) if m else (combined, '')

income_raw[['COUNTYNAME', 'TOWNNAME']] = income_raw['鄉鎮市區'].apply(
    lambda x: pd.Series(split_township(x))
)

print('Unique combined:', income_raw['鄉鎮市區'].nunique())
print('Unique COUNTYNAME:', income_raw['COUNTYNAME'].nunique())
print('Unique TOWNNAME:', income_raw['TOWNNAME'].nunique())
print('\nSplit sample:')
income_raw[['鄉鎮市區','COUNTYNAME','TOWNNAME']].drop_duplicates().head(5)

In [ ]:
from scipy.special import erf

def cv_to_gini(cv_pct):
    """Convert coefficient of variation (%) to Gini via lognormal approximation.
    
    Under lognormal income distribution:
      sigma_log = sqrt(ln(1 + cv²))
      Gini = erf(sigma_log / 2)
    """
    cv = cv_pct / 100.0
    sigma_log = np.sqrt(np.log1p(cv ** 2))
    return erf(sigma_log / 2)

# Apply village-level Gini conversion before aggregating
income_raw['gini_village'] = income_raw['變異係數'].apply(cv_to_gini)

township_income = (
    income_raw
    .groupby(['COUNTYNAME', 'TOWNNAME'])
    .apply(
        lambda g: pd.Series({
            'median_household_income': np.average(g['中位數'],     weights=g['納稅單位(戶)']),
            'mean_household_income':   np.average(g['平均數'],     weights=g['納稅單位(戶)']),
            'gini_index':              np.average(g['gini_village'], weights=g['納稅單位(戶)']),
            'tax_household_count':     g['納稅單位(戶)'].sum(),
        }),
        include_groups=False
    )
    .reset_index()
)

print('Township income rows:', len(township_income))
print('\nDescriptive stats:')
print(township_income[['median_household_income', 'gini_index']].describe().round(4))
print(f'\ngini_index range: {township_income.gini_index.min():.4f} – {township_income.gini_index.max():.4f}')

## Step 2 — 低收入戶資料：衛福部統計

**來源**：`1.1.2低收入戶戶數及人數按鄉鎮市區別分(2015~)1150310.xlsx`  
**使用 sheet**：`2020`（CE 年，對應民國 109 年）  
**使用欄位**：Q4 低收入戶**戶數**（欄 56，年底存量，= 最具代表性的年度值）  

**解析策略**：Excel 資料為層級結構（縣市 → 鄉鎮），  
透過追蹤目前所在縣市，將每個鄉鎮正確分配到對應縣市。  

**已知名稱異動**（依實際行政升格修正）：
- `頭份鎮`（苗栗縣）→ `頭份市`（2016 年升格）
- `員林鎮`（彰化縣）→ `員林市`（2015 年升格）

In [ ]:
# Q4 households column: Q1 starts at col 2, each quarter = 18 cols
# Q4 Sub-Total households = col 2 + 3*18 = 56
Q4_COL = 56

COUNTIES_22 = {
    '新北市', '臺北市', '桃園市', '臺中市', '臺南市', '高雄市',
    '基隆市', '新竹市', '嘉義市',
    '新竹縣', '苗栗縣', '彰化縣', '南投縣', '雲林縣', '嘉義縣',
    '屏東縣', '宜蘭縣', '花蓮縣', '臺東縣', '澎湖縣', '金門縣', '連江縣'
}

# Name fixes: poverty data uses old names (before municipality upgrade)
NAME_FIXES = {
    '頭份鎮': '頭份市',   # 苗栗縣：2016 年升格
    '員林鎮': '員林市',   # 彰化縣：2015 年升格
}

poverty_raw = pd.read_excel(
    POVERTY_XLS, sheet_name='2020', header=None, skiprows=7,
    usecols=[0, Q4_COL]
)
poverty_raw.columns = ['name_cn', 'q4_households']
poverty_raw = poverty_raw.dropna(how='all')
poverty_raw['name_cn'] = poverty_raw['name_cn'].astype(str).str.strip()

print('Raw poverty rows:', len(poverty_raw))
poverty_raw.head(8)

In [ ]:
records = []
current_county = None

for _, row in poverty_raw.iterrows():
    name  = row['name_cn']
    count = row['q4_households']

    if name in ('總計', 'nan') or str(count) == 'nan':
        continue
    if name in COUNTIES_22:
        current_county = name
        continue  # county sub-total — skip
    if current_county:
        # Apply name fixes for upgraded townships
        town = NAME_FIXES.get(name, name)
        records.append({
            'COUNTYNAME': current_county,
            'TOWNNAME':   town,
            'poverty_family_below_poverty_level': int(count)
        })

poverty_df = pd.DataFrame(records)
print('Township poverty records:', len(poverty_df))
poverty_df.head(5)

## Step 3 — 載入人口資料（含 TOWNCODE 與總戶數）

In [ ]:
pop_df = pd.read_csv(POP_CSV)
pop_df['total_household_count'] = (
    pop_df['population'] / pop_df['average_household_size']
).round().astype(int)

print('Population rows:', len(pop_df))
print('Columns:', pop_df.columns.tolist())
pop_df[['TOWNCODE', 'COUNTYNAME', 'TOWNNAME', 'population', 'total_household_count']].head(5)

## Step 4 — 合併：收入 + 人口 + 貧窮 → 輸出

In [ ]:
# Left merge from population (368 rows) → income
income_merged = pop_df[['TOWNCODE', 'COUNTYNAME', 'TOWNNAME', 'total_household_count']].merge(
    township_income[['COUNTYNAME', 'TOWNNAME', 'median_household_income',
                      'mean_household_income', 'gini_index', 'tax_household_count']],
    on=['COUNTYNAME', 'TOWNNAME'],
    how='left'
)

match_rate = income_merged['median_household_income'].notna().mean()
print(f'Income merge rows: {len(income_merged)}')
print(f'Match rate: {match_rate:.1%}')

unmatched = income_merged[income_merged['median_household_income'].isna()]
if len(unmatched) > 0:
    print('\nUnmatched:')
    print(unmatched[['COUNTYNAME','TOWNNAME']].to_string())

In [ ]:
# Merge poverty with TOWNCODE and compute rate
poverty_merged = poverty_df.merge(
    pop_df[['TOWNCODE', 'COUNTYNAME', 'TOWNNAME', 'total_household_count']],
    on=['COUNTYNAME', 'TOWNNAME'],
    how='left'
)
poverty_merged['poverty_family_below_poverty_level_rate'] = (
    poverty_merged['poverty_family_below_poverty_level']
    / poverty_merged['total_household_count']
)

match_rate_pov = poverty_merged['TOWNCODE'].notna().mean()
print(f'Poverty merge rows: {len(poverty_merged)}')
print(f'Match rate: {match_rate_pov:.1%}')

unmatched_pov = poverty_merged[poverty_merged['TOWNCODE'].isna()]
if len(unmatched_pov) > 0:
    print('\nUnmatched:')
    print(unmatched_pov[['COUNTYNAME','TOWNNAME']].to_string())

In [ ]:
output = income_merged.merge(
    poverty_merged[['TOWNCODE', 'poverty_family_below_poverty_level',
                    'poverty_family_below_poverty_level_rate']],
    on='TOWNCODE',
    how='left'
)

OUTPUT_COLS = [
    'TOWNCODE',
    'COUNTYNAME',
    'TOWNNAME',
    'median_household_income',
    'gini_index',
    'poverty_family_below_poverty_level_rate',
    'poverty_family_below_poverty_level',
    'tax_household_count',
]

output[OUTPUT_COLS].to_csv(OUT_PATH, index=False, encoding='utf-8-sig')
print(f'Saved: {OUT_PATH.resolve()}')
print(f'Shape: {output.shape}  (expected: 368 rows)')
print('\nMissing values:')
print(output[OUTPUT_COLS].isna().sum())

## Step 5 — 驗證

### 5a. 數值範圍檢查

In [ ]:
out = pd.read_csv(OUT_PATH)

print('=== 輸出統計 ===')
print(f'筆數: {len(out)} (應為 368)')
print()
print(out[['median_household_income', 'gini_index',
           'poverty_family_below_poverty_level_rate']].describe().round(4))

print()
print('驗證條件:')
print(f'  貧窮率範圍 [0.003, 0.12]: {((out["poverty_family_below_poverty_level_rate"] >= 0.003) & (out["poverty_family_below_poverty_level_rate"] <= 0.12)).all()}')
print(f'  所得 > 0 (所有筆): {(out["median_household_income"] > 0).all()}')
print(f'  Gini > 0 (所有筆): {(out["gini_index"] > 0).all()}')

### 5b. 所得地理梯度：前後 10 名

In [ ]:
out_full = out.merge(pop_df[['TOWNCODE','COUNTYNAME','TOWNNAME']], on='TOWNCODE', how='left')

display_cols = ['COUNTYNAME', 'TOWNNAME', 'median_household_income', 'gini_index',
                'poverty_family_below_poverty_level_rate']

print('=== 所得最高 10 鄉鎮（預期：台北、新竹科技走廊） ===')
print(out_full.nlargest(10, 'median_household_income')[display_cols].to_string(index=False))

print('\n=== 所得最低 10 鄉鎮（預期：偏遠農村） ===')
print(out_full.nsmallest(10, 'median_household_income')[display_cols].to_string(index=False))

### 5c. 縣市層級交叉驗證

將鄉鎮資料彙整至縣市，與**主計總處「109年平均每戶家庭收支」**的縣市可支配所得排名比較，  
確認兩者排序方向一致（Spearman 相關）。

In [ ]:
from scipy.stats import spearmanr

# Aggregate township median → county median (weighted by tax_household_count)
county_check = (
    out_full
    .groupby('COUNTYNAME')
    .apply(
        lambda g: np.average(g['median_household_income'], weights=g['tax_household_count']),
        include_groups=False
    )
    .rename('income_agg')
    .sort_values(ascending=False)
    .reset_index()
)

print('=== 縣市加權平均所得排名 ===')
print(county_check.to_string(index=False))

print()
print('預期排序：臺北市 > 新竹市/縣 >> 多數縣市 >> 離島/農業縣')

### 5d. 分布圖

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(out['median_household_income'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('家戶中位數所得分布\n(千元/年)', fontsize=11)
axes[0].set_xlabel('中位數所得 (千元)')

axes[1].hist(out['gini_index'], bins=30, color='orange', edgecolor='white')
axes[1].set_title('Gini 代理指數分布\n(變異係數)', fontsize=11)
axes[1].set_xlabel('變異係數')

axes[2].hist(out['poverty_family_below_poverty_level_rate'] * 100, bins=30, color='tomato', edgecolor='white')
axes[2].set_title('低收入戶比例分布\n(%)', fontsize=11)
axes[2].set_xlabel('低收入戶比例 (%)')

for ax in axes:
    ax.set_ylabel('鄉鎮數')

plt.suptitle('台灣各鄉鎮收入類特徵分布（109年基準）', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / '04c_taiwan_income_distributions.png', dpi=300, bbox_inches='tight')
plt.show()

## Step 6 — Summary

In [ ]:
out_final = pd.read_csv(OUT_PATH)

print('=== 收入特徵彙整 Summary ===')
print(f'鄉鎮筆數: {len(out_final)} / 368')
print(f'缺值率: {out_final.isna().mean().max():.1%} (各欄最大值)')
print()
print('欄位說明 → DeepSolar 對應:')
mapping = {
    'median_household_income':                '綜稅109年 村里加權中位數 (千元/年) → median_household_income',
    'gini_index':                             '綜稅109年 村里加權變異係數 → gini_index (代理)',
    'poverty_family_below_poverty_level_rate':'衛福部2020 Q4 低收入戶比例 → poverty_family_below_poverty_level_rate',
    'poverty_family_below_poverty_level':     '衛福部2020 Q4 低收入戶戶數 → poverty_family_below_poverty_level',
    'tax_household_count':                    '報稅戶數（QA用，不入模型）',
}
for col, desc in mapping.items():
    print(f'  {col:<48} ← {desc}')
print()
print(f'輸出檔案: {OUT_PATH.resolve()}')